# Speaker-Aligned Transcript on Google Colab

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yeiichi/whisper-smith/blob/main/notebooks/colab_aligned_transcript.ipynb)

This notebook produces a speaker-aligned transcript JSON from an audio file using `whisper-smith`.

**Runtime → Change runtime type → T4 GPU** before running.

**Requirements (Colab Secrets — key icon in sidebar):**
- `OPENAI_API_KEY` — your OpenAI API key
- `HUGGINGFACE_TOKEN` — your Hugging Face token (must have accepted [pyannote/speaker-diarization-community-1](https://huggingface.co/pyannote/speaker-diarization-community-1) terms)

## Step 1 — Install whisper-smith

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

repo_url = "git+https://github.com/yeiichi/whisper-smith.git"
target_dir = Path("/content/whisper-smith-packages")
bin_dir = Path("/content/whisper-smith-bin")
shutil.rmtree(target_dir, ignore_errors=True)
shutil.rmtree(bin_dir, ignore_errors=True)
target_dir.mkdir(parents=True, exist_ok=True)
bin_dir.mkdir(parents=True, exist_ok=True)

install_command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--target",
    str(target_dir),
    "--upgrade",
    f"whisper-smith[colab] @ {repo_url}",
]
completed = subprocess.run(
    install_command,
    capture_output=True,
    text=True,
)
if completed.stdout:
    print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
completed.check_returncode()

target_path = str(target_dir)
if target_path not in sys.path:
    sys.path.insert(0, target_path)

launcher = bin_dir / "whisper-smith"
launcher.write_text(
    "#!/usr/bin/env python3\n"
    "import sys\n"
    f"sys.path.insert(0, {target_path!r})\n"
    "from whisper_smith.cli import main\n"
    "raise SystemExit(main())\n",
    encoding="utf-8",
)
launcher.chmod(0o755)
os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"
subprocess.run(["whisper-smith", "--help"], check=True)

## Step 2 — Load credentials from Colab Secrets

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["HUGGINGFACE_TOKEN"] = userdata.get("HUGGINGFACE_TOKEN")

## Step 3 — Upload audio file

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
audio_path = Path(next(iter(uploaded)))
output_path = audio_path.with_suffix(".aligned.json")
print(f"Uploaded: {audio_path}")

## Step 4 — Run the aligned transcript pipeline

This is equivalent to running the following CLI command locally:

```bash
whisper-smith audio.m4a --align --diarization-model pyannote/speaker-diarization-community-1 --output audio.aligned.json
```

In [ ]:
!whisper-smith "{audio_path}" --align --diarization-model pyannote/speaker-diarization-community-1 --output "{output_path}"

## Step 5 — Preview results

In [ ]:
import json

data = json.loads(output_path.read_text(encoding="utf-8"))
for seg in data["segments"]:
    speaker = seg.get("speaker") or "UNKNOWN"
    print(f"[{seg['start']:6.2f}s – {seg['end']:6.2f}s]  {speaker:12s}  {seg['text'].strip()}")

## Step 6 — Download the JSON

In [ ]:
files.download(str(output_path))

---
## Advanced — Explicit GPU pipeline

For fine-grained control (e.g. specifying number of speakers), load the
pyannote pipeline manually, move it to the GPU, and pass it via `pipeline=`.
This also avoids re-downloading the model if you call `diarize_audio` multiple times.

In [ ]:
import torch
from pyannote.audio import Pipeline
from whisper_smith.align import assign_speakers
from whisper_smith.diarize import diarize_audio
from whisper_smith.exporters import export_json
from whisper_smith.transcribe import transcribe_audio

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

diarize_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=os.environ["HUGGINGFACE_TOKEN"],
)
diarize_pipeline.to(device)

transcript  = transcribe_audio(audio_path)
diarization = diarize_audio(audio_path, pipeline=diarize_pipeline, num_speakers=2)
aligned     = assign_speakers(transcript, diarization)

output_path.write_text(export_json(aligned), encoding="utf-8")
print(f"Saved → {output_path}")